# Node/Edge to ROI Matrix Test

This notebook tests the `node_edge_to_roi_matrix` function which converts BrainNet Viewer format node/edge files into a full ROI x ROI connectivity matrix.

In [ ]:
import sys
import os

# Add parent directory to path so we can import HarrisLabPlotting
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import pandas as pd
from HarrisLabPlotting import (
    load_node_file,
    load_edge_file,
    node_edge_to_roi_matrix,
    load_mesh_file,
    create_brain_connectivity_plot
)

print("Imports successful!")

## 1. Define File Paths

In [ ]:
# File paths
node_file = r"C:\Users\Azad Azargushasb\Research\HarrisLabPlotting\test_files\node and edges\total.node"
edge_file = r"C:\Users\Azad Azargushasb\Research\HarrisLabPlotting\test_files\node and edges\total(1).edge"
roi_coords_file = r"C:\Users\Azad Azargushasb\Research\roi_coordinates\atlas_170_coordinates\atlas_170_coordinates_comma.csv"
mesh_file = r"C:\Users\Azad Azargushasb\Research\brain_filled_3_smoothed.gii"

print(f"Node file: {node_file}")
print(f"Edge file: {edge_file}")
print(f"ROI coords file: {roi_coords_file}")
print(f"Mesh file: {mesh_file}")

## 2. Test Individual Loading Functions

In [ ]:
# Test load_node_file
node_df = load_node_file(node_file)
print(f"Node file loaded: {len(node_df)} nodes")
print(f"\nNode DataFrame columns: {list(node_df.columns)}")
print(f"\nFirst 5 nodes:")
node_df.head()

In [ ]:
# Test load_edge_file
edge_matrix = load_edge_file(edge_file)
print(f"Edge matrix loaded: {edge_matrix.shape}")
print(f"Non-zero edges: {np.count_nonzero(edge_matrix)}")
print(f"Min value: {edge_matrix.min():.4f}")
print(f"Max value: {edge_matrix.max():.4f}")

In [ ]:
# Load ROI reference file to see structure
roi_df = pd.read_csv(roi_coords_file)
print(f"ROI reference file loaded: {len(roi_df)} ROIs")
print(f"\nColumns: {list(roi_df.columns)}")
print(f"\nFirst 5 ROIs:")
roi_df.head()

## 3. Test node_edge_to_roi_matrix Function

In [ ]:
# Convert node/edge files to full ROI matrix
full_matrix, roi_names, node_indices = node_edge_to_roi_matrix(
    node_file=node_file,
    edge_file=edge_file,
    roi_reference=roi_coords_file,
    roi_name_column='roi_name'
)

print(f"Full matrix shape: {full_matrix.shape}")
print(f"Number of ROIs in reference: {len(roi_names)}")
print(f"Number of nodes mapped: {len(node_indices)}")
print(f"Non-zero edges in full matrix: {np.count_nonzero(full_matrix)}")

In [ ]:
# Show which ROIs from the node file were mapped
print("Mapped node ROIs and their indices in the full matrix:")
for i, (node_name, full_idx) in enumerate(zip(node_df['roi_name'], node_indices)):
    print(f"  Node {i}: {node_name} -> Full matrix index {full_idx}")

In [ ]:
# Verify the mapping is correct by comparing a few values
print("Verification - comparing original edge values to mapped values:")
print("-" * 60)

# Find some non-zero edges to verify
nonzero_i, nonzero_j = np.where(edge_matrix != 0)
for k in range(min(5, len(nonzero_i))):
    i, j = nonzero_i[k], nonzero_j[k]
    orig_val = edge_matrix[i, j]
    mapped_i, mapped_j = node_indices[i], node_indices[j]
    mapped_val = full_matrix[mapped_i, mapped_j]
    print(f"Edge ({i},{j}): {orig_val:.4f} -> Full matrix ({mapped_i},{mapped_j}): {mapped_val:.4f} ✓")

## 4. Test Error Handling

In [ ]:
# Test with a ROI list that doesn't contain all node ROIs (should raise error)
incomplete_roi_list = ['ROI_A', 'ROI_B', 'ROI_C']  # Missing the actual ROI names

try:
    node_edge_to_roi_matrix(
        node_file=node_file,
        edge_file=edge_file,
        roi_reference=incomplete_roi_list
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print("Correctly raised ValueError for missing ROIs:")
    print(f"\n{str(e)[:500]}...")

## 5. Visualize with Brain Connectivity Plot

In [ ]:
# Load mesh
vertices, faces = load_mesh_file(mesh_file)
print(f"Mesh loaded: {len(vertices)} vertices, {len(faces)} faces")

In [ ]:
# IMPORTANT: Reload roi_df from the SAME file used for matrix conversion
# This ensures the DataFrame indices match the full_matrix dimensions
roi_df_for_plot = pd.read_csv(roi_coords_file)
print(f"ROI DataFrame for plotting: {len(roi_df_for_plot)} ROIs")
print(f"Full matrix shape: {full_matrix.shape}")
print(f"Dimensions match: {len(roi_df_for_plot) == full_matrix.shape[0]}")

# Create visualization using the full matrix and matching ROI coordinates
fig, stats = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=roi_df_for_plot,  # Use the DataFrame that matches the matrix size
    connectivity_matrix=full_matrix,
    edge_threshold=0,  # Show all non-zero edges
    mesh_opacity=0.3,
    node_size=8,
    node_color='purple',
    node_border_color='magenta'
)

print(f"\nGraph Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

fig.show()

## 6. Summary

In [ ]:
print("=" * 60)
print("TEST SUMMARY")
print("=" * 60)
print(f"Node file: {len(node_df)} nodes loaded")
print(f"Edge file: {edge_matrix.shape} matrix loaded")
print(f"ROI reference: {len(roi_names)} ROIs")
print(f"Output matrix: {full_matrix.shape}")
print(f"Nodes successfully mapped: {len(node_indices)}/{len(node_df)}")
print(f"Non-zero edges preserved: {np.count_nonzero(full_matrix)}")
print("=" * 60)
print("All tests passed!")